In [7]:
from langchain.chat_models import ChatOpenAI
from dotenv import load_dotenv, find_dotenv
from langchain.tools import Tool
from langchain.agents.agent_types import AgentType
from langchain_experimental.sql import SQLDatabaseChain
from langchain.utilities import SQLDatabase

from sqlalchemy.engine import URL
import os

In [5]:
load_dotenv(find_dotenv())

True

In [8]:
# Create connection string
connectionURL = URL.create(
    "postgresql+psycopg2",
    host=os.environ.get("HOST"),
    database="enqurious_etl_db",
    username=os.environ.get("USER"),
    port=5432,
    password=os.environ.get("PASSWORD"),
)

In [9]:
llm = ChatOpenAI(temperature=0, model='gpt-3.5-turbo-16k')

In [10]:
db = SQLDatabase.from_uri(connectionURL)
db_chain = SQLDatabaseChain.from_llm(db=db, llm=llm)

In [31]:
tools = [
    Tool(
        name='Local DB',
        func=db_chain.run,
        description="Useful when we need to answer the questions about users"
    )
]

In [32]:
# break prompt into a prefix and suffix the prefix is our instructions
prefix = """Have a conversation with a human, answering the following
questions as best as you can based on the memory available and the context provided 
below. If the question cannot be answered using the information provided, answer with 
"I don't know".
"""

# and the suffix our user input and output indicator
# The agent_scratchpad is where we add every thought or action the agent has already performed.
# All thoughts and actions (within the current agent executor chain) can then be accessed 
# by the next thought-action-observation loop, enabling continuity in agent actions.
suffix = """Begin!"

{chat_history}
Question: {input}
{agent_scratchpad}"""

In [33]:
from langchain.prompts import PromptTemplate
from langchain.agents import ZeroShotAgent

In [35]:
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["input", "chat_history", "agent_scratchpad"],
)

In [36]:
from langchain.memory import ConversationBufferMemory

In [37]:
memory = ConversationBufferMemory(
    memory_key="chat_history"
)

In [38]:
from langchain.chains import LLMChain

In [39]:
llm_chain = LLMChain(llm=llm, prompt=prompt)

In [40]:
from langchain.agents import initialize_agent, ZeroShotAgent

In [41]:
agent = ZeroShotAgent(
    tools=tools,
    llm_chain=llm_chain,
    verbose=True
)

In [42]:
from langchain.agents import AgentExecutor

In [45]:
agent_chain = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True, memory=memory
)

In [47]:
agent_chain.run("How many tables are there in the database?")



> Entering new AgentExecutor chain...
Thought: I need to check the Local DB database to find the number of tables.
Action: Local DB
Action Input: Get table count
Observation: Final answer here: There are 14 tables in the database.
Thought:

Retrying langchain.chat_models.openai.ChatOpenAI.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised RateLimitError: Rate limit reached for default-gpt-3.5-turbo-16k in organization org-bopjHrMEteRaVQsQwATYgdCP on requests per min. Limit: 3 / min. Please try again in 20s. Contact us through our help center at help.openai.com if you continue to have issues. Please add a payment method to your account to increase your rate limit. Visit https://platform.openai.com/account/billing to add a payment method..
Retrying langchain.chat_models.openai.ChatOpenAI.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised RateLimitError: Rate limit reached for default-gpt-3.5-turbo-16k in organization org-bopjHrMEteRaVQsQwATYgdCP on requests per min. Limit: 3 / min. Please try again in 20s. Contact us through our help center at help.openai.com if you continue to have issues. Please add a payment method to your account to increase your rate limit. 

I now know the final answer
Final Answer: There are 14 tables in the database.

> Finished chain.


'There are 14 tables in the database.'